## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [1]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


# 07 — Multi-stage pipeline with worker-labeled CSVs

This tutorial demonstrates a **3-stage pipeline** where the LLM agent head
dynamically routes data through different processing stages based on data
characteristics. Each stage processes data from a **single CSV file per worker**,
maintaining the pattern from Tutorial 06.

## Pipeline architecture

Each worker gets its own CSV file (`worker{number}.csv`) and processes it fully.
The head coordinates which workers participate in which stage and what
transformation to apply.

## Stage assignments (example)

| Stage | Purpose | Workers | Worker CSV |
|-------|---------|---------|-------------|
| 1. Ingestion | Raw sensor preprocessing | worker1, worker2, worker3 | `worker1.csv`, `worker2.csv`, `worker3.csv` |
| 2. Feature extraction | Aggregation & smoothing | worker1, worker2, worker3 | `worker1.csv`, `worker2.csv`, `worker3.csv` |
| 3. Prediction | Run inference | worker4, worker5 | `worker4.csv`, `worker5.csv` |

## Parameters

In [2]:
import os, time, json, subprocess, random

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4,worker5").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
expecting     : ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']
fleet managed : Gustavo


## Generate each worker's private data

One CSV per worker with synthetic sensor data for pipeline processing.

In [3]:
DATA_DIR = os.environ.get("DATA_DIR", "/data")

def write_worker_csv(path, values):
    with open(path, "w") as f:
        f.write("value\n")
        for v in values:
            f.write(f"{v}\n")

rng = random.Random(20260930)
os.makedirs(DATA_DIR, exist_ok=True)

for i, node in enumerate(WORKER_NODES, start=1):
    values = [round(rng.gauss(20 + 2 * i, 1.5 + 0.2 * i), 2) for _ in range(12)]
    path = f"{DATA_DIR}/{node}.csv"
    write_worker_csv(path, values)
    print(f"{node}: wrote {len(values)} readings to {path}")

_DESCRIPTION = "Sensor readings held by this worker. One numeric column: value."

# --- local config for each worker -------------------------------------
# This sprint retired CSV_PATH. A worker no longer gets a path from its
# environment: it reads its own ~/.scarlet/config.yaml, profiles whatever
# sources it finds there at boot, and writes its own SQL against them.
#
# HOME for each worker points into this shared volume (see the compose
# file), so the path below is the worker's *real* ~/.scarlet/config.yaml -
# there is no SCARLET_LOCAL_CONFIG override in play.
import yaml as _yaml

for _node in WORKER_NODES:
    _csv = f"{DATA_DIR}/{_node}.csv"
    _scarlet_dir = os.path.join(DATA_DIR, _node, ".scarlet")
    os.makedirs(_scarlet_dir, exist_ok=True)
    with open(os.path.join(_scarlet_dir, "config.yaml"), "w") as _fh:
        _yaml.safe_dump({"sources": [{
            "name": f"{_node}_readings",
            "type": "csv",
            "mode": "local",
            "description": _DESCRIPTION,
            "path": _csv,
        }]}, _fh, sort_keys=False)
print(f"wrote {len(WORKER_NODES)} worker configs under {DATA_DIR}/<worker>/.scarlet/")


worker1: wrote 12 readings to /data/worker1.csv
worker2: wrote 12 readings to /data/worker2.csv
worker3: wrote 12 readings to /data/worker3.csv
worker4: wrote 12 readings to /data/worker4.csv
wrote 4 worker configs under /data/<worker>/.scarlet/


## Bring the fleet up

In [4]:
AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]

def compose(*args):
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )
    
    print("bringing up:", ", ".join(AGENT_SERVICES))
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

Gustavo owns this fleet - not launching anything.
If the wait below times out, check the Gustavo deployment.


## Wait for the fleet to come online

In [5]:
from scarlets.messaging import Messenger

FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

fleet online: ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']


{'__composer_api__',
 'loop-agent-head_head',
 'loop-agent-worker_worker1',
 'loop-agent-worker_worker2',
 'loop-agent-worker_worker3',
 'loop-agent-worker_worker4',
 'notebook_probe'}

## Ask the LLM head to orchestrate the pipeline

In this simplified version, each worker has its own CSV with all needed data,
and the head coordinates which workers run which stage. The head can decide
to use `sum` with different transforms across the workers' CSV data.

In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# Deliberately terse. The endpoint sits behind a 120s Cloudflare proxy read
# timeout, and the head calls the model with every skill's tool definition
# attached - so a long, multi-clause prompt pushes a single turn past that
# limit and the whole request dies with a 524 before any skill is dispatched.
# Measured: this prompt's earlier six-sentence form took >120s with tools
# attached and never dispatched anything; one sentence per stage completes.
#
# The stages are also expressed in terms the harness can actually execute.
# The earlier form asked stage 2 to read /data/{worker}_processed.csv, but
# CSV_PATH is one path per container fixed at deploy time and no skill takes
# a path argument - so that stage could never have run, whatever the model
# decided.
answer = await ask_scarlet_agent(
    "Run a 3-stage pipeline over the online workers. "
    "Stage 1: sum with transform=identity. "
    "Stage 2: sum with transform=square. "
    "Stage 3: combine those into mean and variance. "
    "Report each stage's result separately."
)
print(answer)

Here are the results of the 3-stage pipeline:

**Stage 1: Sum (transform=identity)**
- Sum of values: 1200.4
- Number of rows: 48
- Columns: 1

**Stage 2: Sum of Squares (transform=square)**
- Sum of squared values: 30461.2374
- Number of rows: 48
- Columns: 1

**Stage 3: Derived Statistics**
- Mean: 25.008333333333336
- Variance: 9.19237638888876

The pipeline successfully computed the mean and variance across all 48 rows from the 4 online workers.


## Teardown

In [7]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")

Gustavo owns this fleet - leaving it alone.
